In [0]:
%sql
DROP TABLE IF EXISTS catalog_ventas.gold.kpi_top_productos;

CREATE OR REPLACE TABLE catalog_ventas.gold.kpi_top_productos
USING DELTA
AS

WITH base AS (
  SELECT
      p.descrip,
      p.categoria,
      SUM(f.total) AS facturacion,
      SUM(f.cant)  AS unidades,
      
      COUNT(DISTINCT f.id_venta) AS tickets

  FROM catalog_ventas.gold.fact_ventas f

  LEFT JOIN catalog_ventas.gold.dim_producto p
    ON f.id_articulo = p.id_articulo 

  GROUP BY p.descrip, p.categoria
),

ranking AS (
  SELECT
  *,
  RANK() OVER (ORDER BY facturacion DESC) AS rnk
  FROM base
)

SELECT * FROM ranking WHERE rnk <= 10

In [0]:
%sql
SELECT * FROM catalog_ventas.gold.kpi_top_productos

In [0]:
%sql
/*
El granel lidera ampliamente el ranking:
1.	1 kilo
2.	1/4 kilo
3.	1/2 kilo
Estos tres formatos concentran:
•	Mayor facturación
•	Mayor cantidad de tickets
•	Mayor volumen de unidades

•	El kilo es el producto estrella
•	Los bombones, palitos y tortas funcionan como productos de impulso
•	Los formatos familiares acompañan el consumo grupal

*/